In [1]:
import json
import time
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd
import requests

# Konfigurasi URL Overpass API dan Header dengan identitas Anda
URL = "https://overpass-api.de/api/interpreter"
HEADERS = {"User-Agent": "INSTIKI-BIFP243-Praktikum/1.0 (2501010315@instiki.ac.id)"}

# Query Overpass API final untuk Kecamatan Kintamani
QUERY = """
[out:json][timeout:90];
area["name"="Kintamani"]["boundary"="administrative"]["admin_level"="6"]->.kec;
nwr["tourism"~"^(hotel|guest_house|hostel|motel|apartment|chalet)$"](area.kec);
out center tags;
"""

# Fungsi akuisisi dengan mekanisme Retry (mengatasi HTTP status 429 / 504)
def ambil_overpass(query, maks_percobaan=4, jeda=5):
    for percobaan in range(1, maks_percobaan + 1):
        try:
            r = requests.post(URL, data={"data": query}, headers=HEADERS, timeout=120)
            if r.status_code == 200:
                return r.json()
            print(f"[percobaan {percobaan}] status {r.status_code}, tunggu {jeda}s")
        except (requests.Timeout, requests.ConnectionError) as e:
            print(f"[percobaan {percobaan}] {type(e).__name__}, tunggu {jeda}s")
        time.sleep(jeda)
        jeda *= 2
    raise RuntimeError("Gagal mengambil data dari Overpass API")

In [2]:
# Eksekusi pengambilan data dari Overpass API
data = ambil_overpass(QUERY)

# Buat folder dan simpan respons JSON mentah dengan stempel waktu
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
Path("data/raw").mkdir(parents=True, exist_ok=True)
raw_filename = f"osm_akomodasi_kintamani_{stamp}.json"
raw_path = Path(f"data/raw/{raw_filename}")
raw_path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"✅ Data mentah berhasil disimpan: {raw_path}")
print(f"📊 Jumlah elemen hasil akuisisi: {len(data['elements'])}")

✅ Data mentah berhasil disimpan: data/raw/osm_akomodasi_kintamani_20261111_034814.json
📊 Jumlah elemen hasil akuisisi: 32


In [3]:
# Fungsi perataan data (Flattening JSON ke DataFrame)
def ratakan(elements):
    baris = []
    for e in elements:
        tg = e.get("tags", {})
        pusat = e.get("center", e)  # node: lat/lon langsung; way/relation: di center
        baris.append({
            "osm_type": e["type"],
            "osm_id": e["id"],
            "nama": tg.get("name"),
            "jenis": tg.get("tourism"),
            "lat": pusat.get("lat"),
            "lon": pusat.get("lon"),
            "desa": tg.get("addr:village"),
            "bintang": tg.get("stars"),
            "jumlah_kamar": tg.get("rooms"),
            "website": tg.get("website") or tg.get("contact:website"),
            "url_osm": f"https://www.openstreetmap.org/{e['type']}/{e['id']}",
        })
    return pd.DataFrame(baris)

# Ubah ke DataFrame dan simpan ke file CSV
akomodasi = ratakan(data["elements"])
Path("data/interim").mkdir(parents=True, exist_ok=True)
csv_path = Path("data/interim/akomodasi_kintamani.csv")
akomodasi.to_csv(csv_path, index=False)

print(f"✅ Dataset tabular berhasil disimpan: {csv_path}")
print("\n--- Ringkasan Dataset ---")
print(akomodasi["jenis"].value_counts())
print("\nJumlah akomodasi tanpa nama:", akomodasi["nama"].isna().sum())
display(akomodasi.head())

✅ Dataset tabular berhasil disimpan: data/interim/akomodasi_kintamani.csv

--- Ringkasan Dataset ---
jenis
hotel          16
guest_house     9
hostel          4
chalet          2
motel           1
Name: count, dtype: int64

Jumlah akomodasi tanpa nama: 1


,osm_type,osm_id,nama,jenis,lat,lon,desa,bintang,jumlah_kamar,website,url_osm
0,node,1977779907,The Lakeview Hotel Bali,hotel,-8.283140,115.365844,None,None,None,http://lakeviewbali.com/en/,https://www.openstreetmap.org/node/1977779907
1,node,2107226701,Arlina,guest_house,-8.249815,115.398465,None,None,None,None,https://www.openstreetmap.org/node/2107226701
2,node,2107227001,Toya Devasya Resort,hotel,-8.250164,115.399798,None,None,None,None,https://www.openstreetmap.org/node/2107227001
3,node,4150326329,Guesthouse,hotel,-8.281456,115.362223,None,None,None,None,https://www.openstreetmap.org/node/4150326329
4,node,4297310696,The Cave Hotel,hotel,-8.280748,115.362048,None,2,5,None,https://www.openstreetmap.org/node/4297310696


In [5]:
{
  "endpoint_url": "https://overpass-api.de/api/interpreter",
  "query_file": "docs/query_overpass.txt",
  "acquisition_timestamp_utc": "2026-11-11T04:00:00Z",
  "total_elements": 45,
  "raw_file_name": "osm_akomodasi_kintamani_20261111_120000.json",
  "md5_checksum": "(diisi hash md5 file json setelah diunduh)",
  "licensing": "Open Database License (ODbL) - © OpenStreetMap contributors",
  "collector_name": "Ni Putu Riska Pramesti Cahya Dewi",
  "collector_nim": "2501010315"
}

{'endpoint_url': 'https://overpass-api.de/api/interpreter',
 'query_file': 'docs/query_overpass.txt',
 'acquisition_timestamp_utc': '2026-11-11T04:00:00Z',
 'total_elements': 45,
 'raw_file_name': 'osm_akomodasi_kintamani_20261111_120000.json',
 'md5_checksum': '(diisi hash md5 file json setelah diunduh)',
 'licensing': 'Open Database License (ODbL) - © OpenStreetMap contributors',
 'collector_name': 'Ni Putu Riska Pramesti Cahya Dewi',
 'collector_nim': '2501010315'}